In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
df = pd.read_csv('/kaggle/input/digit-recognizer/train.csv')
data = np.array(df)
np.random.shuffle(data)


In [ ]:
data = data.T

In [ ]:
m, n = data.shape

In [ ]:
X_train = data[1:1000]
X_train = X_train / 255
y_train = data[0]

In [ ]:
X_train.shape

In [ ]:
m

In [ ]:
y_train

# Building the Nueral Network

In [ ]:
X = X_train
Y = y_train
X_train

In [ ]:
import numpy as np

# ------------------ Initialization ------------------
def init():
    W1 = np.random.rand(10, 784) * 0.01
    b1 = np.zeros((10, 1))
    W2 = np.random.rand(10, 10) * 0.01
    b2 = np.zeros((10, 1))
    return W1, b1, W2, b2

# ------------------ Activations ------------------
def ReLU(Z):
    return np.maximum(0, Z)

def ddx_ReLU(Z):
    return (Z > 0).astype(int)

def Softmax(Z):
    expZ = np.exp(Z - np.max(Z, axis=0, keepdims=True))  # numerical stability
    return expZ / np.sum(expZ, axis=0, keepdims=True)

# ------------------ Forward Propagation ------------------
def for_prop(W1, b1, W2, b2, X):
    Z1 = np.dot(W1, X) + b1
    A1 = ReLU(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = Softmax(Z2)
    return Z1, A1, Z2, A2

# ------------------ One-Hot Encoding ------------------
def one_hot(Y):
    m = Y.size
    one_hot_Y = np.zeros((10, m))
    one_hot_Y[Y, np.arange(m)] = 1
    return one_hot_Y

# ------------------ Backpropagation ------------------
def back_prop(A1, A2, Z1, Z2, W2, X, Y):
    m = X.shape[1]
    Y_onehot = one_hot(Y)

    # Output layer
    dZ2 = A2 - Y_onehot                 # (10, m)
    dW2 = (1/m) * np.dot(dZ2, A1.T)     # (10, 10)
    db2 = (1/m) * np.sum(dZ2, axis=1, keepdims=True)  # (10,1)

    # Hidden layer
    dZ1 = np.dot(W2.T, dZ2) * ddx_ReLU(Z1)  # (10, m)
    dW1 = (1/m) * np.dot(dZ1, X.T)          # (10, 784)
    db1 = (1/m) * np.sum(dZ1, axis=1, keepdims=True)  # (10,1)

    return dW1, db1, dW2, db2

# ------------------ Parameter Update ------------------
def correction(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha):
    W1 -= alpha * dW1
    b1 -= alpha * db1
    W2 -= alpha * dW2
    b2 -= alpha * db2
    return W1, b1, W2, b2

# ------------------ Prediction & Accuracy ------------------
def get_prediction(A2):
    return np.argmax(A2, axis=0)

def get_accuracy(predictions, Y):
    return np.sum(predictions == Y) / Y.size

# ------------------ Gradient Descent ------------------
def gradient_descent(X, Y, iterations, alpha):
    W1, b1, W2, b2 = init()
    m = X.shape[1]

    for i in range(iterations):
        # Forward pass
        Z1, A1, Z2, A2 = for_prop(W1, b1, W2, b2, X)

        # Backward pass
        dW1, db1, dW2, db2 = back_prop(A1, A2, Z1, Z2, W2, X, Y)

        # Update parameters
        W1, b1, W2, b2 = correction(W1, b1, W2, b2, dW1, db1, dW2, db2, alpha)

        # Progress printing
        if i % 10 == 0:
            predictions = get_prediction(A2)
            acc = get_accuracy(predictions, Y)
            print(f"Iteration {i} - Accuracy: {acc:.4f}")

    return W1, b1, W2, b2


In [ ]:
W1,b1,W2,b2 = gradient_descent(X,Y,500,0.1)